# W2 — RAG + LangChain/LangGraph

**오늘 흐름**
1. **셋업** (5분)
2. **Part 1 — RAG** (~1h 50분) — keyword + vector 두 retriever + grounding + 둘 다 RAG 비교
3. **Part 2 — LangChain/LangGraph** (~40분) — W1 Agent + W2 RAG 한 줄 wrap

진행: cell 한 개씩 같이 실행 → 출력 확인 → 다음. **학생이 직접 짤 일 = 0**.

## 0. 셋업

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"

print("준비 완료. 모델:", MODEL)

In [ ]:
# 모델 살아있는지 한 번 호출
resp = client.models.generate_content(model=MODEL, contents="한 줄로 자기소개해줘.")
print(resp.text)

In [ ]:
# notes/ 폴더에 뭐 있는지 확인
NOTES_DIR = Path("../week1/notes")
for p in sorted(NOTES_DIR.glob("*.md")):
    print(p.name)

---

## Part 1 — RAG (Retrieval-Augmented Generation)

### 1-1. LLM 의 할루시네이션 문제

**할루시네이션 (hallucination)** — LLM 이 모르는 내용을 그럴듯하게 지어내는 현상.

**왜 발생하나**
- LLM 은 인터넷 일반 데이터로 학습 → 우리 회사 내부 문서 / 노트는 본 적 없음
- 모르는 걸 물어봐도 "모릅니다" 라고 안 하고 그럴듯한 추측 / 일반론으로 답함
- 모델이 거짓말하려는 게 아니라, **모르는 걸 채워서 답하는 본성**

**왜 실무에서 문제인가**
- 회사 정책 / 사내 매뉴얼 / 제품 사양 같은 내부 정보를 그대로 신뢰할 수 없음
- 그럴듯해 보여서 검증 없이 쓰면 잘못된 결정으로 이어짐

먼저, 회사 정책을 그냥 물어보면 어떤 답이 나오는지 확인해 보자.

In [ ]:
# 노트 안 보여주고 그냥 물어보기
question = "신입사원은 연차 며칠 받아?"
resp = client.models.generate_content(model=MODEL, contents=question)
print(resp.text)

위 답이 바로 **할루시네이션의 예시**. 우리 `policy_leave.md` 의 실제 정책과 다를 가능성 큼.

### 1-2. 해결 방법들

LLM 할루시네이션을 줄이는 대표적 접근:

1. **Fine-tuning** — 우리 데이터로 모델을 재학습. 효과 크지만 비싸고 느림. 데이터 바뀔 때마다 재학습.
2. **system_instruction** — 답 형식 / 추측 금지 등을 강제. RAG 와 함께 씀.
3. **RAG (Retrieval-Augmented Generation)** — 답하기 전에 관련 문서를 찾아서 LLM 에게 같이 줌. 가볍고 빠름. **오늘 다룰 방법.**

### 1-3. RAG = Retrieval-Augmented Generation

> "LLM 한테 그냥 묻지 말고,
> **답에 필요한 문서를 프롬프트에 같이 넣어서** 답하게 하자."

- **R**etrieval (**검색**) — 답에 필요한 문서 골라옴
- **A**ugmented (**증강**) — 그 문서를 프롬프트에 **추가**
- **G**eneration (**생성**) — 보강된 프롬프트로 답 생성

> **핵심은 "찾는 행동" 이 아니라 "프롬프트에 같이 넣는 패턴".**
> 찾는 건 누가 하든 (코드 / Agent) 무방 — RAG 의 본질은 **주입**.

### 1-4. 가장 단순한 RAG — system_instruction 에 노트 박기

In [ ]:
# 1. 노트 한 파일 읽기
policy_text = (NOTES_DIR / "policy_leave.md").read_text(encoding="utf-8")
print(policy_text)

In [ ]:
# 2. system_instruction 에 그 내용 박고 다시 질문
config = types.GenerateContentConfig(
    system_instruction=(
        "당신은 사내 정책에 답하는 도우미입니다. "
        "아래 회사 정책 문서 내용에 근거해서만 답하세요. "
        "문서에 없으면 '문서에 없습니다' 라고 답하세요.\n\n"
        f"=== 회사 정책 ===\n{policy_text}"
    ),
)

resp = client.models.generate_content(model=MODEL, contents=question, config=config)
print(resp.text)

답이 정확해짐. **이게 grounding** — 모델 답변이 우리가 준 문서에 묶임.

근데 노트가 10개, 100개라면? 다 `system_instruction` 에 박으면 토큰 낭비 + 노이즈.

→ 필요한 것만 골라오자 = **`retrieve` 함수**.

### 1-5. retrieve — query 와 매칭되는 노트 골라오기

`retrieve` 함수의 단계를 **셀 하나씩** 풀어서 동작 확인 → 마지막에 함수로 묶는다.

In [ ]:
# === [1] 질문 → 키워드 분할 ===
query = "신입사원인데 연차 언제부터 쓸 수 있어?"
keywords = query.lower().split()
print(keywords)

In [ ]:
# === [2] 한 노트 하나만 매칭 시연 — policy_leave.md ===
path = NOTES_DIR / "policy_leave.md"
text = path.read_text(encoding="utf-8")

score = sum(1 for kw in keywords if kw in text.lower())
print(f"{path.name}: score={score}")
print(f"text 앞 100자: {text[:100]}...")

In [ ]:
# === [3] 모든 노트 돌면서 score 계산 + 누적 ===
results = []
for path in NOTES_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")
    score = sum(1 for kw in keywords if kw in text.lower())
    if score > 0:
        results.append({
            "score": score,
            "filename": path.name,
            "text": text,
            "preview": text[:200],
        })

print(f"매칭된 노트 {len(results)} 개:")
for r in results:
    print(f"  score={r['score']}  {r['filename']}")

In [ ]:
# === [4] score 높은 순으로 정렬 + 상위 3개 ===
results.sort(key=lambda x: -x["score"])
top_3 = results[:3]

for r in top_3:
    print(f"score={r['score']}  {r['filename']}")
    print(f"  preview: {r['preview'][:80]}...")
    print()

In [ ]:
# === [5] 위 4단계를 retrieve 함수로 묶기 ===
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """RAG 의 Retrieval — query 와 매칭되는 노트들 점수순 반환 (전통적 keyword retriever)."""
    keywords = query.lower().split()
    results = []
    for path in NOTES_DIR.glob("*.md"):
        text = path.read_text(encoding="utf-8")
        score = sum(1 for kw in keywords if kw in text.lower())
        if score > 0:
            results.append({
                "score": score,
                "filename": path.name,
                "text": text,
                "preview": text[:200],
            })
    results.sort(key=lambda x: -x["score"])
    return results[:top_k]

In [ ]:
# 다른 질문 — 같은 함수, 다른 결과
for r in retrieve("재택"):
    print(f"  score={r['score']}  {r['filename']}")
    print(f"  preview: {r['preview'][:80]}...")
    print()

이제 위 `retrieve` 를 RAG 흐름에 결합.

질문 → `retrieve` 로 관련 노트 골라옴 → 그 내용을 `system_instruction` 에 박음 → 1번 호출로 답.

= **1-pass RAG**. 루프 없음. 코드가 "검색 먼저" 라고 박아둠.

### 1-6. RAG 완성형 — 단계별로 셀 하나씩

In [ ]:
# === [1] grounding 시스템 프롬프트 템플릿 ===
# {context} 자리에 검색 결과를 넣음.
SYSTEM_INSTRUCTION_TEMPLATE = (
    "당신은 사내 노트 검색 도우미입니다. "
    "아래 [관련 노트] 의 전체 내용을 끝까지 꼼꼼히 읽고, "
    "질문과 관련된 정보 (예: '## 신입사원', '## 자주 묻는 질문' 같은 섹션 포함) 를 찾아 답하세요. "
    "문서에 있는 내용은 명확히 답하고, 없는 내용만 '문서에 없습니다' 라고 답하세요.\n\n"
    "=== 관련 노트 ===\n{context}"
)

In [ ]:
# === [2] Retrieval — query 로 retrieve 호출 ===
question = "신입사원인데 연차 언제부터 쓸 수 있어?"
results = retrieve(question)

print(f"매칭 {len(results)} 개")
for r in results:
    print(f"  {r['filename']} (score={r['score']})")

In [ ]:
# === [3] Augmented — 결과 노트를 system_instruction 에 박기 ===
# (fallback: text 가 없으면 preview 사용 — 옛 retrieve 메모리에 남아도 동작)
context = "\n\n".join(
    f"# {r['filename']}\n{r.get('text') or r.get('preview', '')}"
    for r in results
)

print(context[:500], "...")

In [ ]:
# === [4] Generation — system_instruction + 질문 → 답 ===
config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION_TEMPLATE.format(context=context)
)
resp = client.models.generate_content(model=MODEL, contents=question, config=config)
print(resp.text)

In [ ]:
# === [5] 위 3단계 (Retrieval → Augmented → Generation) 를 rag_answer 함수로 묶기 ===
def rag_answer(question: str) -> str:
    # 1. Retrieval
    results = retrieve(question)
    context = "\n\n".join(
        f"# {r['filename']}\n{r.get('text') or r.get('preview', '')}"
        for r in results
    )
    # 2. Augmented
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION_TEMPLATE.format(context=context)
    )
    # 3. Generation
    resp = client.models.generate_content(model=MODEL, contents=question, config=config)
    return resp.text

In [ ]:
print(rag_answer("재택 신청 절차 알려줘."))

In [ ]:
# 문서에 없는 질문 — grounding 효과 ('문서에 없습니다')
print(rag_answer("우리 회사 식당 메뉴 뭐야?"))

### 1-7. Vector retriever — 실제 만들어보기

지금까지 `retrieve` = keyword 매칭. 한 발 더 나가 **embedding + cosine** 으로 의미 기반 검색을 직접 만들어 본다.

In [ ]:
# === [1] embedding 한 번 — vector 가 어떻게 생겼나 ===
EMBED_MODEL = "gemini-embedding-001"

resp = client.models.embed_content(
    model=EMBED_MODEL,
    contents="쉬는 날",
)
sample_vec = resp.embeddings[0].values

print(f"vector 차원: {len(sample_vec)}")
print(f"앞 5개 값  : {[round(v, 4) for v in sample_vec[:5]]}")

In [ ]:
# === [2] embed 함수 — SDK 호출 단순 wrap ===
def embed(text: str) -> list[float]:
    resp = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return resp.embeddings[0].values

In [ ]:
# === [3] cosine 함수 — 두 vector 의 의미 유사도 ===
import numpy as np

def cosine(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# === [4] 의미 비슷한 단어 vector 가 가깝다 — 검증 ===
v_rest = embed("쉬는 날")
v_yeon = embed("연차")
v_food = embed("점심 메뉴")

print(f"쉬는 날 ↔ 연차      : {cosine(v_rest, v_yeon):.3f}  ← 의미 가까움")
print(f"쉬는 날 ↔ 점심 메뉴 : {cosine(v_rest, v_food):.3f}  ← 의미 멈")

In [ ]:
# === [5] 한 노트 embedding 해보기 ===
text = (NOTES_DIR / "policy_leave.md").read_text(encoding="utf-8")
vec = embed(text)
print(f"vec 차원: {len(vec)}, 앞 3개: {[round(v, 3) for v in vec[:3]]}")

In [ ]:
# === [6] 모든 노트 embedding 해서 note_db (in-memory vector store) 만들기 ===
note_db = []
for path in NOTES_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")
    note_db.append({
        "filename": path.name,
        "text": text,
        "vec": embed(text),
    })

print(f"note_db: {len(note_db)} 개 노트 저장")
for n in note_db:
    print(f"  {n['filename']:30s}  vec 차원={len(n['vec'])}")

In [ ]:
# === [7] 한 질문 vector vs note_db — cosine 점수 계산 ===
q = "쉬는 날"
q_vec = embed(q)

for n in note_db:
    sim = cosine(q_vec, n["vec"])
    print(f"  {sim:.3f}  {n['filename']}")

In [ ]:
# === [8] 점수 정렬 + 상위 3개 ===
scored = []
for n in note_db:
    sim = cosine(q_vec, n["vec"])
    scored.append({
        "score": sim,
        "filename": n["filename"],
        "preview": n["text"][:200],
        "text": n["text"],
    })
scored.sort(key=lambda x: -x["score"])

for r in scored[:3]:
    print(f"  {r['score']:.3f}  {r['filename']}")

In [ ]:
# === [9] 위 단계를 retrieve_vector 함수로 묶기 ===
def retrieve_vector(query: str, top_k: int = 3) -> list[dict]:
    q_vec = embed(query)
    scored = []
    for n in note_db:
        sim = cosine(q_vec, n["vec"])
        scored.append({
            "score": sim,
            "filename": n["filename"],
            "preview": n["text"][:200],
            "text": n["text"],
        })
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]

In [ ]:
# === [10] keyword retrieve vs vector retrieve_vector — '쉬는 날' 로 비교 ===
print("=== keyword retrieve('쉬는 날') ===")
kw = retrieve("쉬는 날")
if not kw:
    print("  (매칭 0개 — '쉬는 날' 이 노트에 없음)")
for r in kw:
    print(f"  score={r['score']}  {r['filename']}")

print("\n=== vector retrieve_vector('쉬는 날') ===")
for r in retrieve_vector("쉬는 날"):
    print(f"  score={r['score']:.3f}  {r['filename']}")

**관전 포인트**:
- keyword 검색: "쉬는 날" 단어가 노트에 없으면 0건
- vector 검색: "연차" 가 있는 `policy_leave.md` 도 **의미상 가까워서 잡아냄**

> **R-A-G 골격은 동일.** retriever 본문만 keyword → vector.

⚠️ **embedding 모델 404 NOT_FOUND 폴백** — SDK 버전에 따라:
- 우선 시도: `gemini-embedding-001` (현재 셀에서 사용 중)
- 폴백 1: `models/text-embedding-004` (full path)
- 폴백 2: `import google.generativeai as legacy` + `legacy.embed_content(model="models/text-embedding-004", content=text)["embedding"]`
- 또는 외부 모델: `sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2`)

### 1-8. Vector RAG — `retrieve_vector` 로 같은 RAG 흐름

`rag_answer` 의 첫 줄 (`results = retrieve(question)`) 만 `retrieve_vector(question)` 으로 바꾸면 vector 기반 RAG 완성.
**R-A-G 골격은 그대로, retriever 만 교체.**

In [ ]:
def rag_answer_vector(question: str) -> str:
    results = retrieve_vector(question)  # ★ keyword retrieve → vector retrieve_vector 만 변경
    context = "\n\n".join(
        f"# {r['filename']}\n{r.get('text') or r.get('preview', '')}"
        for r in results
    )
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION_TEMPLATE.format(context=context)
    )
    resp = client.models.generate_content(model=MODEL, contents=question, config=config)
    return resp.text

In [ ]:
# 같은 질문, keyword RAG vs vector RAG — '쉬는 날' 은 노트에 없는 단어
q = "신입사원이 쉬는 날 얼마나 쓸 수 있어?"

print("=== keyword RAG (retrieve) ===")
print(rag_answer(q))
print()
print("=== vector RAG (retrieve_vector) ===")
print(rag_answer_vector(q))

### 1-9. 정리

- **RAG = Retrieval + Augmented Generation**
- **R**etrieval (검색) = `retrieve(query)` (keyword) 또는 `retrieve_vector(query)` (vector)
- **A**ugmented **G**eneration (증강 + 생성) = `system_instruction` 에 그 노트 박고 1번 호출 → 답
- **1-pass.** 루프 없음. 코드가 "검색 먼저" 박은 형태.
- retriever 본문만 바꾸면 됨 — **R-A-G 골격은 동일.**
- 다음 (Part 2) = 같은 일을 **LangChain/LangGraph** 로 더 짧게.

---

## Part 2 — LangChain / LangGraph

**지금까지 직접 짠 것**:
- **W1 (지난주)** = Agent (도구 호출 + 루프) ~50줄
- **W2 Part 1 (오늘)** = RAG (`retrieve` + `system_instruction` grounding) ~20줄

→ 둘 다 합쳐 ~70줄 짠 코드.

**LangChain (LLM/도구 추상화) + LangGraph (agent/workflow 표준)** 으로 가면 같은 일을 ~5줄로.
Agent + RAG 가 한 번에 wrap.

### W1 + W2 = LangChain/LangGraph 한 줄로

| 우리가 직접 짠 것 | LangChain / LangGraph |
|---|---|
| 도구 JSON 스키마 (W1, ~15줄) | `@tool` 1줄 (LangChain) |
| agent loop (W1, ~30줄) | `create_react_agent(...)` 1줄 (**LangGraph** prebuilt) |
| `rag_answer_vector` (W2, vector 검색 + grounding) | `find_notes` 도구 한 줄 (Part 1 의 `retrieve_vector` 재활용) |

> `create_react_agent` 가 **LangGraph 의 prebuilt** — 즉 우리가 이미 LangGraph 를 쓰고 있는 셈.
> LangGraph = agent / workflow 의 **state machine 표준** (요즘 실무 표준).

In [ ]:
# === LangChain 부품 imports ===
# LangChain = LLM / 도구 / 프롬프트 추상화. 아직 agent loop 는 안 들어옴.
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# === LangChain — @tool 데코레이터로 함수를 LLM 도구로 등록 ===
# Part 1 의 retrieve_vector 를 LangChain 도구로 wrap. 본문 한 줄.
@tool
def find_notes(query: str) -> list[dict]:
    """notes 폴더에서 query 와 의미적으로 가까운 노트들을 점수순으로 반환한다.

    회사 정책·회의록·온보딩 등 사내 문서에 대한 질문에 사용.
    Vector 기반 (embedding + cosine 유사도) — 단어 일치 아닌 의미 매칭.
    """
    return retrieve_vector(query, top_k=3)

In [ ]:
# === LangChain — LLM 객체 (Gemini wrap) ===
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
)

In [ ]:
# === LangChain 만으로 chat 한 번 — llm.invoke + SystemMessage/HumanMessage ===
# 도구 X. 그냥 Gemini 한테 메시지 보내기. (W1 의 chat 함수와 동일한 일.)
messages = [
    SystemMessage(content="당신은 사내 노트 검색 도우미입니다."),
    HumanMessage(content="신입사원이 연차 며칠 받아?"),
]
resp = llm.invoke(messages)
print(resp.content)

In [ ]:
# === LangChain 만으로 RAG 한 번 — retrieve + llm.invoke ===
# rag_answer 와 동일하지만 Gemini SDK 대신 LangChain 의 llm.invoke 사용.
question = "신입사원이 쉬는 날 얼마나 쓸 수 있어?"
results = retrieve_vector(question)
context = "\n\n".join(
    f"# {r['filename']}\n{r.get('text') or r.get('preview', '')}"
    for r in results
)

messages = [
    SystemMessage(content=SYSTEM_INSTRUCTION_TEMPLATE.format(context=context)),
    HumanMessage(content=question),
]
resp = llm.invoke(messages)
print(resp.content)

**여기까지 = LangChain 만**

- `@tool` 로 도구 등록 (스키마 자동), `llm.invoke(messages)` 로 LLM 호출
- 근데 **LLM 이 도구를 자동 호출 안 함** — 위 셀에서 우리가 직접 `retrieve_vector` 호출 후 결과를 system_instruction 에 박았음
- = LangChain 단독으론 W1 의 agent_loop 못 만듦. **`for turn in range(MAX_TURNS):` 직접 짜야 함.**

**이제부터 LangGraph** — LLM 이 도구 호출 / 루프 / 메시지 누적까지 알아서.

### LangGraph 시작

In [ ]:
# === LangGraph imports ===
# LangGraph = agent / workflow 의 state machine. LangChain 부품을 받아서 굴림.
#
# ⚠️ NOTE: LangGraph 1.2 부터 create_react_agent 는 langchain.agents.create_agent 로 이동.
# (docstring 에 deprecation warning 표시.) 동작은 동일. 학습용으로는 이 함수 그대로 사용.
# 마이그레이션: pip install langchain + `from langchain.agents import create_agent` 로 교체.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langgraph.prebuilt import create_react_agent

In [ ]:
# === System prompt — Agent 에게 줄 지시 (도구 호출 자동) ===
AGENT_PROMPT = (
    "당신은 사내 노트 검색 도우미입니다. "
    "정책·규정·회의록 질문은 find_notes 도구로 검색해서 답하세요. "
    "도구 결과에 근거해서만 답하고, 없는 내용은 '문서에 없습니다' 라고 답하세요."
)

In [ ]:
# === LangGraph — create_react_agent (Agent loop state machine, prebuilt) ===
# ★ LangChain 의 llm + @tool 을 LangGraph 가 받아서 state machine 으로 굴림.
# LLM 이 직접 도구 호출 여부 / 루프 / 메시지 누적 결정 — W1 의 agent_loop 자동화.
agent = create_react_agent(llm, tools=[find_notes], prompt=AGENT_PROMPT)
print("agent 준비 완료")

In [ ]:
result = agent.invoke({"messages": [("user", "신입사원인데 연차 언제부터 쓸 수 있어?")]})

# 메시지 흐름 전체 출력 — LangGraph 가 LLM ↔ 도구 ↔ LLM 자동 굴림
for msg in result["messages"]:
    msg.pretty_print()

### 한 발 더 — LangGraph `StateGraph` 로 RAG workflow 직접 짜기

`create_react_agent` = LangGraph 의 **prebuilt** (편의 함수).
실무는 보통 `StateGraph` 로 **노드 + 엣지** 를 직접 정의해서 워크플로우를 그림.

#### 사실 단순 RAG 면 함수 한 개로 충분

`rag_answer` 처럼 `retrieve → generate` 만이면 StateGraph 안 써도 됨.
**LangGraph 의 진가는 흐름이 복잡할 때**:

```
[질문]
  ↓
[retrieve] → "결과 충분?" → No → [웹검색] → 합치기
  ↓ Yes                                ↓
[generate] ← ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ┘
  ↓
[사람 검수 대기]
  ↓
[최종 답]
```

- **조건부 분기** ("결과 충분?") — 함수 chain 의 if/else 가 그래프로 명시화
- **루프** (검수 → 부족하면 retrieve 로 다시) — recursion 직접 안 짜도 됨
- **HITL (Human-in-the-Loop)** — 노드에서 사람 승인 대기
- **체크포인트** — 중간 상태 저장 후 재시작
- **병렬** — 여러 retriever 동시 실행 후 합치기 (map-reduce)

> 함수 chain 으론 if/else + recursion 지옥. 그래프로 짜면 명확.
> 아래 코드 = **단순 retrieve → generate** 미리보기 (노드 2개 + 엣지 3개).

In [ ]:
# === [1] state 정의 — 각 노드 사이로 흐를 데이터 ===
from typing import TypedDict

class RAGState(TypedDict):
    question: str
    context: str
    answer: str

In [ ]:
# === [2] retrieve 노드 — state['question'] → state['context'] ===
def retrieve_node(state: RAGState) -> dict:
    results = retrieve_vector(state["question"], top_k=3)
    context = "\n\n".join(
        f"# {r['filename']}\n{r.get('text') or r.get('preview', '')}"
        for r in results
    )
    return {"context": context}

In [ ]:
# === [3] generate 노드 — state['context'] → state['answer'] ===
def generate_node(state: RAGState) -> dict:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION_TEMPLATE.format(context=state["context"])
    )
    resp = client.models.generate_content(model=MODEL, contents=state["question"], config=config)
    return {"answer": resp.text}

In [ ]:
# === [4] 그래프 조립 — 노드 + 엣지 ===
from langgraph.graph import StateGraph, START, END

graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

rag_workflow = graph.compile()
print("workflow 컴파일 완료")

In [ ]:
# === [5] 실행 — agent.invoke 와 비슷한 모양 ===
result = rag_workflow.invoke({"question": "신입사원이 쉬는 날 얼마나 쓸 수 있어?"})
print(result["answer"])

**정리 — LangGraph 의 두 가지 패턴**:

| | `create_react_agent` (prebuilt) | `StateGraph` (직접) |
|---|---|---|
| 형태 | **Agent** — LLM 이 도구 호출 결정 | **Workflow** — 코드가 흐름 정의 |
| 쓸 때 | 자유로운 멀티턴 에이전트 | 정해진 단계 (RAG · 검수 · 분기 등) |
| 줄 수 | 1줄 wrap | 노드 함수 + 엣지 ~15줄 |

> **언제 어느 걸?**
> - 단순 RAG (`retrieve → generate`) = 함수 한 개로도 충분 (StateGraph 굳이 X)
> - 자유로운 챗봇 / 에이전트 = `create_react_agent`
> - 복잡한 흐름 (분기 / 루프 / HITL / 체크포인트 / 병렬) = `StateGraph`
> - 실무에선 둘 다 섞어 씀 (예: StateGraph 안의 한 노드가 create_react_agent).

**마무리 메시지**:

- Agent 의 도구 호출 / 결과 / 답변은 LangGraph 가 알아서 `messages` 에 누적
- 프레임워크 = **같은 골격을 한 줄 wrapper 로 묶은 것** (마법 X)
- 골격 (W1 + W2 Part 1) 을 알았으니 LangChain / LangGraph 코드도 그냥 읽힘.